# Clasificación Binaria usando **Logistic Regression**

## Introducción

La Logistic Regression es uno de los métodos más utilizados **para problemas de clasificación**. Mientras que la linear regression usa una línea (o un **hiperplano** con tantas dimensiones como variables independientes) para predecir el **valor continuo** de la variable de respuesta, la logistic regression ajusta una curva para **predecir la probabilidad** de que la variable de resultado pertenezca a una categoría particular o no.

Es importante enfatizar que, aunque se llama "regresión", **la logistic regression se usa principalmente para resolver problemas de clasificación, no de regresión**. El nombre proviene del hecho de que modela la relación entre las features y el **log-odds** (o logit) del resultado usando una función lineal — lo cual es una forma de regresión. Sin embargo, transforma esta combinación lineal a través de la **función sigmoide** para producir probabilidades entre 0 y 1. La predicción final se realiza aplicando un umbral (típicamente 0.5) a estas probabilidades para asignar una etiqueta de clase.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

## *Dataset*

En este ejemplo, analizaremos datos de 100 estudiantes que solicitaron admisión a una universidad específica. El proceso de selección consiste en dos exámenes de ingreso y una segunda fase de entrevistas personales. No tenemos datos que nos permitan cuantificar el desempeño de los estudiantes en la entrevista personal, pero sí tenemos sus puntuaciones de examen y si fueron admitidos o no. Consideraremos por tanto dos variables independientes (las puntuaciones de los exámenes) y una variable dependiente (admitido o no admitido).

In [ ]:
df = pd.read_csv('data/ex2data1.txt', names=['exam1', 'exam2', 'admitted'])
df.head()

,exam1,exam2,admitted
0,34.623660,78.024693,No
1,30.286711,43.894998,No
2,35.847409,72.902198,No
3,60.182599,86.308552,Yes
4,79.032736,75.344376,Yes


## Logistic Regression con una sola variable independiente

Para simplificar aún más, **analizaremos inicialmente la posibilidad de ser admitido basándonos en una única variable independiente (la puntuación en el primer examen de ingreso)**.

In [ ]:
df.plot(kind='scatter',  # Crear un diagrama de dispersión
        x='exam1',       # con 'exam1' en el eje x
        xlabel='Puntuación en el Examen 1',
        y='admitted',    # y 'admitted' en el eje y
        ylabel='Admisión',
        figsize=(6, 2),  # Reducir la altura del gráfico
        title='Admisión Universitaria en Función de la Puntuación del Examen 1');

Al mostrar todos los datos en un diagrama de dispersión, podemos observar que los estudiantes admitidos generalmente tienen puntuaciones más altas que los no admitidos, pero la relación no es lineal, ya que la variable dependiente solo puede tomar dos valores (admitido o no admitido) y no valores continuos.

Aun así, podemos dar a estos datos una interpretación probabilística, entendiendo que la probabilidad de ser admitido aumenta a medida que aumenta la puntuación del examen. De esta manera, podemos buscar una curva que se ajuste a los datos y nos permita predecir la probabilidad de ser admitido en función de la puntuación del examen. Una probabilidad de 1 significaría que el estudiante es admitido con total certeza, mientras que una probabilidad de 0 significaría que el estudiante no es admitido en absoluto.

Para ello, necesitamos convertir la variable dependiente en una variable continua que represente la probabilidad, y dado que los datos que tenemos son reales a posteriori, su probabilidad es conocida: los que fueron admitidos ```(df['admitted'] == 'Yes')``` tienen una probabilidad de 1, mientras que los que no fueron admitidos (df['admitted'] == 'No') tienen una probabilidad de 0.

In [ ]:
# Modificar los valores de la columna 'admitted' para que sean 0 y 1
df['admitted'] = df['admitted'].map({'No': 0, 'Yes': 1})
# df['admitted'] = df['admitted'].replace('No', 0).replace('Yes', 1) # Forma alternativa de hacerlo
df.head()

En estos casos, la función sigmoide nos permite obtener una curva que se ajusta a los datos, siendo plana en los extremos (probabilidad 0 o 1) y con una pendiente más pronunciada en el centro, dejando una zona de transición entre aquellos donde la probabilidad de ser admitido es alta o muy baja.

$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

In [ ]:
def sigmoid(x): # Función sigmoide
    return 1 / (1 + np.exp(-x))

x = np.arange(-10, 10, 0.1) # Array de ejemplo para la función sigmoide

plt.figure(figsize=(6, 2))
plt.plot(x, sigmoid(x))
plt.title('Función Sigmoide')
plt.show()

Esta es la función que la logistic regression usa para convertir la combinación lineal de entradas en una probabilidad.

**Cómo funciona:** La logistic regression encuentra los mejores parámetros (pesos) usando **estimación de máxima verosimilitud** — ajusta iterativamente los pesos para maximizar la probabilidad de observar los datos de training reales. Scikit-learn usa algoritmos de optimización como L-BFGS o SAG para encontrar estos parámetros óptimos.

Usaremos la clase ```LogisticRegression``` de ```sklearn.linear_model``` para ajustar el modelo a nuestros datos:

In [ ]:
from sklearn.linear_model import LogisticRegression
# max_iter aumenta el número máximo de iteraciones para la convergencia
model = LogisticRegression(max_iter=1000).fit(df[['exam1']], df['admitted'])

```df[['exam1']]``` es un DataFrame con solo la columna de puntuaciones del primer examen. Es importante recordar que el método fit de la clase LogisticRegression espera un DataFrame con las variables independientes (ya que normalmente habrá más, cada una en su propia columna), y no una Series. Por eso usamos ```df[['exam1']]``` en lugar de ```df['exam1']```.

Con ese modelo ajustado, podemos obtener predicciones para un nuevo valor, usando el método ```predict``` de la misma forma que con la linear regression:

In [ ]:
plt.figure(figsize=(6, 3))
plt.scatter(df['exam1'], df['admitted'])
new_scores = pd.DataFrame({'exam1': [55, 60]}) # Crear un nuevo DataFrame con dos puntuaciones
plt.scatter(new_scores, model.predict(new_scores), color='red', marker='X', s=200)
plt.show()

Podemos observar cómo el modelo predice que un estudiante con una puntuación de 55 no será admitido, pero uno con una puntuación de 60 sí lo será. El modelo está calculando dónde está el punto de corte de la curva sigmoide para determinar si la probabilidad de ser admitido es mayor o menor que 0.5, pero podemos tener más información si en lugar de la predicción directa, lo que obtenemos es la probabilidad de ser admitido, usando el método `predict_proba`.

In [ ]:
print(model.classes_) # Clases detectadas por el modelo

proba_new_scores = model.predict_proba(new_scores)
print(proba_new_scores)

print(proba_new_scores[:,0]+proba_new_scores[:,1]) # La suma de las dos columnas es 1

proba_admit_new_scores = model.predict_proba(new_scores)[:,1]

`predict_proba` devuelve un array con tantas columnas como posibles valores de la variable dependiente (las categorías). En este caso las categorías son 0 (no admitido) y 1 (admitido), como podemos comprobar en el atributo `classes_` del modelo, por lo que el array tendrá dos columnas, una con la probabilidad de ser admitido y otra con la probabilidad de no ser admitido (que será su complemento, 1 - probabilidad de ser admitido).

Por lo tanto, para este caso, la información de las dos columnas es redundante, ya que si la probabilidad de ser admitido es del 80%, la probabilidad de no ser admitido será del 20%, pero normalmente tendremos más de dos categorías, por lo que es importante tener en cuenta que la primera columna del array es la probabilidad de la primera categoría, la segunda columna es la probabilidad de la segunda categoría, etc.

In [ ]:
plt.figure(figsize=(6, 3))
plt.scatter(df['exam1'], df['admitted'])
print(new_scores)
print(proba_admit_new_scores)
plt.scatter(new_scores, proba_admit_new_scores, color='red', marker='X', s=200)

for i, txt in enumerate(np.round(proba_admit_new_scores, 2)): # Para cada valor de proba_admit_new_scores, tomar su índice y su valor redondeado
    plt.hlines(y=proba_admit_new_scores[i], xmin=30, xmax=new_scores['exam1'][i], linestyle='--') # Dibujar línea horizontal desde el eje
    plt.annotate(txt, (new_scores['exam1'][i], proba_admit_new_scores[i])) # Anotar el valor en el punto

plt.show()

Creando un array de múltiples valores posibles para X y generando sus probabilidades de ser admitido, podemos representar la curva sigmoide que se ajusta a los datos:

In [ ]:
x_sigmoid = np.linspace(30, 100, 1000) # Array de 1000 valores entre 30 y 100 (límites del eje x que estamos usando)
df_sigmoid = pd.DataFrame({'exam1': x_sigmoid}) # Crear un nuevo DataFrame con esas puntuaciones hipotéticas
y_sigmoid = model.predict_proba(df_sigmoid[['exam1']]) # Calcular los valores y para esas puntuaciones

In [ ]:
plt.figure(figsize=(6, 3))

plt.plot(x_sigmoid, y_sigmoid[:, 1], 'r-', label='Probabilidad de Admisión')

plt.scatter(df['exam1'], df['admitted'])
plt.xlabel('Puntuación en el Examen 1')
plt.ylabel('Admisión Universitaria')
plt.legend()
plt.show()

Adicionalmente podemos mostrar la probabilidad de la otra categoría (no ser admitido) para ver cómo la suma de ambas probabilidades siempre es 1.

In [ ]:
# Adicionalmente podemos mostrar la probabilidad de la otra categoría (no ser admitido) para ver cómo la suma de ambas probabilidades siempre es 1.
plt.figure(figsize=(6, 3))
plt.plot(x_sigmoid, y_sigmoid[:, 0], 'b--', label='Probabilidad de No Admisión') # Probabilidad de no admisión
plt.plot(x_sigmoid, y_sigmoid[:, 1], 'r-', label='Probabilidad de Admisión') # Probabilidad de admisión
plt.scatter(df['exam1'], df['admitted']) # Puntos de datos reales

plt.xlabel('Puntuación en el Examen 1')
plt.ylabel('Admisión Universitaria')
plt.legend()
plt.show()

## Logistic Regression con múltiples variables independientes

El enfoque normal será calcular esta regresión con ambas puntuaciones de examen, ya que eso es lo que tenemos. En este caso, la curva sigmoide será tridimensional, ya que la probabilidad de ser admitido dependerá de dos variables independientes.

Podemos hacer una exploración inicial de los datos visualizándolos con un diagrama de dispersión en el que el color de los puntos indica si el estudiante fue admitido o no.

In [ ]:
scatter = plt.scatter(df['exam1'], df['exam2'], c=df['admitted']) 
plt.xlabel("Puntuación en el Examen 1")
plt.ylabel("Puntuación en el Examen 2")
plt.legend(scatter.legend_elements()[0], ['No Admitido', 'Admitido'])
plt.show()

In [ ]:
fig = plt.figure()
ax = plt.axes(projection='3d')

x = df['exam1']
y = df['exam2']
z = df['admitted']

scatter = ax.scatter(x, y, z, c=z, cmap='viridis')

ax.set_xlabel('Puntuación en el Examen 1')
ax.set_ylabel('Puntuación en el Examen 2')
ax.set_zlabel('Admitido')

plt.show()

Como era de esperar, los estudiantes admitidos tienden a tener puntuaciones más altas en ambos exámenes que los no admitidos, pero la relación no es lineal, por lo que la logistic regression es un buen método para ajustar una curva a estos datos.

Si queremos usar más de una variable independiente, el proceso es el mismo, pero en lugar de usar un DataFrame con una sola columna, usaremos un DataFrame con tantas columnas como variables independientes tengamos.

In [ ]:
X = df.drop(columns='admitted') # Devolver dataframe sin la columna 'admitted'
y = df['admitted']

model_multi = LogisticRegression(max_iter=1000).fit(X, y)

# Nota: Estas predicciones usan el modelo de una sola variable del paso anterior
# Para usar el modelo multivariable, necesitaríamos ambas puntuaciones de examen
print(f"Predicciones del modelo con un examen (probabilidades): {model.predict_proba(new_scores)[:,1]}")
print(f"Predicciones del modelo con un examen (clases): {model.predict(new_scores)}")

En lugar de la curva 3D, podemos simplemente representar en 2D el límite de decisión (***decision boundary***), que es el lugar donde la probabilidad de ser admitido es del 50%.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

_, ax = plt.subplots()
DecisionBoundaryDisplay.from_estimator( # Crear un gráfico con el decision boundary
    model_multi,
    X,
    cmap='Paired',
    ax=ax,
    response_method="predict",
    plot_method="pcolormesh",
    shading="auto",
    xlabel="Puntuación en el Examen 1",
    ylabel="Puntuación en el Examen 2",
    eps=0.5,
)

scatter = plt.scatter(df['exam1'], df['exam2'], c=df['admitted'],
            edgecolors="k",
            cmap=plt.cm.Paired
            ) 

plt.xlabel("Puntuación en el Examen 1")
plt.ylabel("Puntuación en el Examen 2")
plt.legend(scatter.legend_elements()[0], ['No Admitido', 'Admitido'])
plt.show()

## Fuentes y material complementario
- https://aprendeia.com/algoritmo-regresion-logistica-machine-learning-teoria/
- https://cienciadedatos.net/documentos/py17-regresion-logistica-python.html
- https://nbviewer.org/github/jdwittenauer/ipython-notebooks/blob/master/notebooks/ml/ML-Exercise2.ipynb
- https://realpython.com/logistic-regression-python/
- https://cesguiro.es/doku.php?id=clase:ia:saa:2eval:clasificacion_modelos_1